<a href="https://colab.research.google.com/github/lmaas37/FDD-Week-6/blob/main/lats_intranet_exercises_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language Agent Tree Search on Company Intranet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eth-fdd-fs26/FDD-WE6-private/blob/main/05_lats/lats_intranet_exercises_student.ipynb)

*From Data to Solutions, Weekend 6: Language Agent Tree Search.*

A company intranet, one question, twelve clicks. **Play it yourself first.**

Next, you will build the agent that plays the game using a method called **Language Agent Tree Search**. Instead of relying on random guesses to figure out the best move like traditional tree search algorithms do, it uses a standard, pre-trained language model to suggest and grade the possible next moves.

Run the setup cell, then go play.


In [1]:
# @title Setup: install the backend, fetch the intranet, import dependencies { display-mode: "form" }
import os
import subprocess
import sys

try:
    import transformers
except ImportError:
    print("installing transformers (the language model this notebook runs on) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE6-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if os.path.isdir(REPO_NAME):
        print("Repo already present — refreshing to latest…")
        res = subprocess.run(["git", "-C", REPO_NAME, "pull", "-q", url], capture_output=True, text=True)
        if res.returncode != 0:
            print("  (could not pull — using the existing copy)")
    else:
        print("Cloning the exercise repo…")
        res = subprocess.run(["git", "clone", "-q", url], capture_output=True, text=True)
        if res.returncode != 0:
            tail = (res.stderr.strip().splitlines() or ["(no message)"])[-1]
            print("  clone failed:", tail)

# Move to the folder holding lats_intranet/ (05_lats/ in this repo) so imports resolve.
# Local Jupyter is often already there; Colab clones the full repo under REPO_NAME.
_INTRANET = os.path.join("lats_intranet", "intranet.py")
for _root in [
        os.path.join(REPO_NAME, "05_lats"),
        "05_lats",
        ".",
        os.path.dirname(os.getcwd()),
        os.path.join(os.path.dirname(os.getcwd()), "05_lats"),
        os.getcwd()]:
    if os.path.exists(os.path.join(_root, _INTRANET)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        f"lats_intranet not found — the clone of {REPO_OWNER}/{REPO_NAME} did not land "
        "(see the message above). Check the Colab runtime has network access, then re-run this cell.")
ENV_DIR = os.path.join(os.getcwd(), "lats_intranet")
sys.path.insert(0, ENV_DIR)

import intranet
from intranet import (MAX_STEPS, QUERIES, SITE_KEYS, SITES, IntranetEnv, initial_state,
                      token_f1, transition)

N_PAGES = sum(len(SITES[key].pages) for key in SITE_KEYS)
N_LINKS = sum(len(page.links) for key in SITE_KEYS for page in SITES[key].pages)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:                             # torch missing, or a broken install
    DEVICE = "none"

print("Working directory:", os.getcwd(), "· helpers importable ✅")
print("intranet loaded from", ENV_DIR)
print("%d sites, %d pages, %d links, %d questions, budget %d actions"
      % (len(SITE_KEYS), N_PAGES, N_LINKS, len(QUERIES), MAX_STEPS))
if DEVICE == "cuda":
    print("torch sees a GPU, so the language model later on will be comfortable.")
elif DEVICE == "cpu":
    print("torch sees no GPU. The model later on will run, slowly. Runtime, Change")
    print("runtime type, T4 takes a minute and is worth it.")
else:
    print("no torch here, so the model cannot load. There is no stand in: install torch.")


Cloning the exercise repo…
Working directory: /content/FDD-WE6-public/05_lats · helpers importable ✅
intranet loaded from /content/FDD-WE6-public/05_lats/lats_intranet
5 sites, 40 pages, 55 links, 10 questions, budget 12 actions
torch sees a GPU, so the language model later on will be comfortable.


## 0.1 The Intranet Game 🎮

Welcome to **Nordhelm Instruments AG**. Invented company, invented people, real feeling of
being lost on an intranet.

Five sites. Forty pages. You land at the portal with nothing open and a question in your
hand.

**Play it.** The window below is the same environment the agent will get: same page, same
side pane, same twelve actions, same reward at the end. You press a button; later the
agent will write that button as text.

**First Question in the Game:**

> In which room is the coffee lounge of the building where the Vega team sits?

Find the answer that scores **1.000**, in at most twelve actions. Coffee lounges sound like
facilities. Start there. See how far that gets you.

Three things to notice while you click — the rest of the notebook sits on them:

1. **You only see the page you are standing on.** No site map. To find out what is on a page, go stand on it.
2. **Every click costs one of twelve.** One episode can read at most 11 of the 40 pages. The answer counts as a cost too.
3. **Answer in the page's words, and answer short.** The reward is token overlap with the
   span it wanted. An extra word costs a fraction. A wrong word costs everything.


In [2]:
# @title Given: Functions between an agent and the environment { display-mode: "form" }

from browser import show_state, walk, reveal

def step(state, action):
    """GIVEN. One action played from a state: (next state, reward, done, info).

    Nothing is ever rewound. A state is nine plain keys, so it can be kept and walked
    back into exactly, and browsing is read only, so there is nothing to undo. That is
    what will make a tree search over this environment legal at all, later.
    """
    nxt, _, reward, done, info = transition(state, action)
    return nxt, float(reward), bool(done), info


def is_terminal(state):
    """GIVEN. The episode is over: it answered, or it ran out of steps."""
    return bool(state["done"])


def reward_of(state):
    """GIVEN. THE ONLY NUMBER IN THIS NOTEBOOK THAT IS NOT THE MODEL TALKING ABOUT
    ITSELF. Zero until the episode ends."""
    return float(state["reward"])


def answer_of(state):
    """GIVEN. The text that was submitted, or None if nothing was."""
    return state["answer"]


def top_action(state):
    """GIVEN. The policy's single best action here, with no sampling at all.

    `greedy=True` takes the top of the model's own ranking instead of sampling from it,
    which is what makes the agent below greedy. The search later does the opposite.

    The last two lines are the one wrinkle in an otherwise plain loop: WITH ONE ACTION
    OF BUDGET LEFT, ONLY ANSWERS ARE OFFERED. An agent that never answers is not more
    honest than one that answers badly, and without this the baseline spends its twelfth
    action turning a page and scores 0.000 by running out rather than by being wrong.
    """
    last = (MAX_STEPS - state["steps"]) <= 1
    actions = LM.propose(state, n=1, greedy=True, answers_only=last)
    if not actions and last:
        actions = LM.propose(state, n=1, greedy=True)
    if not actions:
        return None
    return actions[0]


print("step, is_terminal, reward_of, answer_of and top_action are loaded")
print()
print("| function     | what it does                                      | inputs         | outputs                          |")
print("|--------------|---------------------------------------------------|----------------|----------------------------------|")
print("| step         | play one action from a state                      | state, action  | (next_state, reward, done, info) |")
print("| is_terminal  | whether the episode is over                       | state          | True if answered or out of steps |")
print("| reward_of    | the environment's score, not the model's          | state          | float; 0 until the episode ends  |")
print("| answer_of    | the text that was submitted                       | state          | the answer string, or None       |")
print("| top_action   | the policy's single best action, no sampling      | state          | an action string, or None        |")


step, is_terminal, reward_of, answer_of and top_action are loaded

| function     | what it does                                      | inputs         | outputs                          |
|--------------|---------------------------------------------------|----------------|----------------------------------|
| step         | play one action from a state                      | state, action  | (next_state, reward, done, info) |
| is_terminal  | whether the episode is over                       | state          | True if answered or out of steps |
| reward_of    | the environment's score, not the model's          | state          | float; 0 until the episode ends  |
| answer_of    | the text that was submitted                       | state          | the answer string, or None       |
| top_action   | the policy's single best action, no sampling      | state          | an action string, or None        |


In [3]:
# @title Play the Intranet Game { display-mode: "form" }
from importlib import reload
import browser
reload(browser)
from browser import explore, html_path

explore(0)

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [4]:
# @title Run if Intranet Game is not working in Notebook { display-mode: "form" }
print("If the window below is empty, open folder icon on the sidebar and download the file:")
print(" ", html_path())
print("Then open the downloaded file intranet_game.html in a browser.")

If the window below is empty, open folder icon on the sidebar and download the file:
  /content/FDD-WE6-public/05_lats/lats_intranet/intranet_game.html
Then open the downloaded file intranet_game.html in a browser.


#### What just happened

So, you clicked around, headed over to `facilities`, and found the *Coffee lounges by building* page. Perfect, right? It has coffee, it has buildings, and it has room codes.

But wait—there’s a catch. The table has four different rows, and the word "Vega" isn't anywhere on the page!

Welcome to the core mechanic of our intranet game. Every question here is a **Two-Hop mystery**:

| 🕵️ The Mystery | 🧩 What you need | 📍 Where to find it |
| :--- | :--- | :--- |
| **Hop 1: The Missing Link** | Find the missing piece of the puzzle (*Which building does the Vega team work in?*) | Over in the `people` section! |
| **Hop 2: The Final Answer** | Match that info to the right table row (*The room code for that specific building*) | Back in `facilities`! |

Neither page answers the question on its own. The final page just gives you a menu of options, and only Hop 1 tells you which one to order! **Reaching the right page is necessary, but it isn't sufficient.** An agent can be 100% sure it's on the right page and still fail the task.

#### The Danger of Guessing

If you just guess a row blindly from the right page, you'll average a measly **0.225** points. But nail the exact row, and you get a perfect **1.000**.

Why the harsh penalty? Because wrong answers don't share any words with the correct one. A confident but incorrect guess gets a flat **0.000**.

Here’s why this is tricky for an AI agent: that *Coffee lounges* table looks like a goldmine. A smart agent might look at it and think, *"Great, I found the coffee lounges, I'm done!"* Nothing on that page tells you that you are missing a crucial fact.

**The only thing in this entire system that knows whether you picked the right row is the reward the environment pays out at the very end.**

***

Want to see the perfect path? Run `reveal(query_idx=0)` in the next cell to print the intended route.


In [5]:
reveal(query_idx=0)

Q1  In which room is the coffee lounge of the building where the Vega team sits?
    gold answer:  N-310
    hop 1:  people, page 3, Team directory
            where the question's description turns into a name: 'Building Nord'
    hop 2:  facilities, page 3, Coffee lounges by building
            a table of 4 rows. The answers it offers: N-310, S-121, W-204, T-018
            without hop 1, standing on it is a 4 way guess


You can try all the other questions in the game and change the `query_idx` and look up the optimal route.

## 0.2 Language Agent Tree Search 🍀

Two hours ago, you built a Monte Carlo tree search (MCTS) for Tic-Tac-Toe. Nine simple squares, a strict list of moves, and a random rollout to play out the game. Easy!

But you probably just realized why that won't cut it here. When you're staring at the coffee lounge table, your next move isn't "square 7." It's clicking a link or typing a search term, and nothing on the screen tells you where the Vega team is hiding! You can't just randomly mash buttons to see if you're on the right page.

Enter **LATS** from the lecture! We're taking that exact same search but applying it to what an **agent** does next. You'll be building this by coding the LATS building blocks one by one .

![LATS FLOWCHART](https://github.com/eth-fdd-fs26/FDD-WE6-public/blob/main/05_lats/lats_intranet/lats_flowchart.jpeg?raw=1)

### Your clicks on Intranet Game compared with LATS Flowchart

One attempt of LATS is one episode of the game you just played, plus the bookkeeping you
did in your head.

| you, just now | LATS |
|---|---|
| First try: you are at the portal. Later tries: pick which unfinished idea to grow. | **1. Selection** · `descend` |
| You look at the buttons. LATS samples *n* of them at once, not one. | **2. Expansion** · `expand` |
| "Facilities looks right for coffee." The model writes that hunch as a number. | **3. Evaluation** · `evaluate` |
| You keep clicking until you type an answer, or run out of steps. | **4. Simulation** · `simulate` |
| A score appears. 0.000 or 1.000. Not a hunch. | `reward_of` — the one number not from the model |
| You remember which sites were worth coming back to. | **5. Backpropagation** · `backprop` |
| "Vega isn't on this page. Try people." | **6. Reflection** · `reflect` |


If you compare this to this morning's Tic-Tac-Toe loop, here is how LATS shakes things up:

* What LATS Adds ➕

    * 🧠 **Evaluation:** Instead of randomly playing the game to the end to test a move, the model just *guesses* its score (`compute_V`). You'll tuck this step right inside your `expand` function.
    * 📝 **Reflection:** When the agent fails, it writes itself a quick "note to self." The next attempt reads this note to avoid making the exact same mistake. This is the *only* thing saved between tries!

* What LATS Drops ➖

    * 🗑️ **The "Untried" List:** Tic-Tac-Toe has a small, strict list of moves. But here, the agent can type *anything*, so there's no set list to cross off! Instead, we just ask the model for a batch of possible actions (`n_actions`) and turn them *all* into new branches at the exact same time.



## 0.3 One Frozen Model, Three Faces 🎭

We'll use one *frozen* language model playing three different roles like a Broadway star. No training required! The exact same model weights handle all three jobs—the only thing that changes is the prompt we feed it.

In pseudocode, you can think of it simply as `LM("some question", state)`. Depending on the question, the model puts on a different hat:

| 🎭 The Role | 💻 The Code | 🤔 The Question it Answers | ⚙️ Where it's used |
| --- | --- | --- | --- |
| 🎯 **The Policy** | `LM.propose(state, n)` | *"What could I do next, from here?"* | 2. Expansion |
| ⚖️ **The Value** | `LM.value(state)` | *"How promising does this path look?"* | 3. Evaluation |
| 🧐 **The Critic** | `LM.reflect(state, reward)` | *"What went wrong here?"* | 6. Reflection |

**The Star of the Show:**
Our actor today is **`Qwen2.5-3B-Instruct`**. It's a public model sitting at about 6 GB, which is why you'll want to make sure you have a GPU runtime selected (go to *Runtime > Change runtime type > T4*).

In [6]:
# @title Load the one frozen model. A minute or two the first time { display-mode: "form" }
from hf_lm import make_lm, MODEL_ID
from agent_lib import Memory
import agent_lib

if DEVICE != "cuda":
    raise RuntimeError(
        "No GPU. " + MODEL_ID + " needs about 12 GB in fp32 on CPU and will crash the\n"
        "session before it finishes loading, which is exactly what happened on 2026-08-12.\n"
        "Runtime, Change runtime type, T4 GPU, Save. It takes a minute.")

LM = make_lm(env=None)          # see MODEL_ID above. No fallback, by design.

print("=" * 62)
print("MODEL ACTUALLY LOADED:", MODEL_ID)      # read this before believing any number
print("=" * 62)
print("loaded:", type(LM).__name__, "on", getattr(LM, "device", "cpu"))


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

hf_lm: loaded Qwen/Qwen2.5-3B-Instruct on cuda (float16)
MODEL ACTUALLY LOADED: Qwen/Qwen2.5-3B-Instruct
loaded: HFLocalLM on cuda


**The three calls, on one state**: Walk two steps into the intranet and ask the model all three questions about where we stand.

In [7]:
state = initial_state(0)
for a in ("goto_site[people]", "next"):
    state = transition(state, a)[0]

print("question:", QUERIES[0].question)
print()
print("propose  ->", LM.propose(state, n=3))
print("value    ->", round(LM.value(state), 3))
print("reflect  ->", LM.reflect(state, 0.0))


question: In which room is the coffee lounge of the building where the Vega team sits?

propose  -> ['answer[Instruments, Embedded Systems and Operations]', 'goto_site[facilities]', 'next']
value    -> 0.0
reflect  -> Start by directly addressing the question about the Vega team's location. Then, mention the team directory for verification, but don't repeat information already provided.


`value` is one number and `reflect` is a sentence. That difference is the whole reason
reflection exists: a scalar says *how* badly it went, words say *why*, and words are what
the next attempt can act on.

---

Now that the setup is done, the problem introduced, and a map in hand of what needs to be done, fill all the blanks below marked with 🎯 and execute the cells. Imported exercise checkers will help you to verify if your solution is correct or not.

## 1. 🎯 Greedy: The Agent to Beat

**What you are about to write.** The simplest agent there is. Ask the model what to do, take the best proposal,
do it, ask again, and stop when the episode ends. No tree, no second opinion, no way back.

This is the **baseline**, the thing the search has to beat, and it is here as a warm up because it is the shortest possible use of the two things everything later is built on, the policy and the environment.

In [8]:
def greedy(query_idx):
    """The agent to beat. One trajectory, the top action every step, no way back."""
    state = initial_state(query_idx)      # the portal, with the question attached
    trajectory = []
    LM.calls = 0                          # so the cost can be printed later

    for _ in range(MAX_STEPS):            # the budget bounds the loop
        if is_terminal(state):
            break
        action = top_action(state)        # given: the policy's single best action
        if action is None:
            break

        # 🎯🎯 Play the action using the step function that takes (state, action) and returns (state, reward, done, info)
        state, reward, done, info = step(state, action)
        # 🎯🎯 Append the action to the trajectory
        trajectory.append(action)

    return {"trajectory": trajectory, "answer": answer_of(state),
            "reward": reward_of(state), "calls": LM.calls}

In [9]:
# @title Checker for exercise 1 { display-mode: "form" }
from checkers import check_greedy
print("checker for exercise 1 loaded")
check_greedy(greedy)

checker for exercise 1 loaded
checking your greedy ...

  OK  it played 5 actions
  OK  replayed from the portal it does the same thing and pays the same 0.000
  OK  it ran to the end of the episode
  OK  it committed to an answer, 'S-121'
  OK  it counted its model calls, 5 of them

PASSED  Exercise, the greedy agent


True

In [10]:
result = greedy(0)

print("greedy trajectory:")
for i, a in enumerate(result["trajectory"], 1):
    print("  %d. %s" % (i, a))
print()
print("answered :", result["answer"])
print("gold     :", QUERIES[0].answer)
print("reward   : %.3f   after %d model calls" % (result["reward"], result["calls"]))


greedy trajectory:
  1. goto_site[facilities]
  2. follow[2]
  3. prev
  4. prev
  5. answer[S-121]

answered : S-121
gold     : N-310
reward   : 0.000   after 5 model calls


It commits to the first plausible looking site, reads the first plausible looking page, and
answers off it. When two sites look equally good it can ping pong between them until the
budget runs out, which is worth watching in the trace: an argmax policy has no mechanism
for noticing it is stuck in a loop.

**It fails for one reason, and it is not that the model is too small. It fails because it
never considers a second option.** Everything from here is machinery for considering more
than one, and it starts with the node that holds one option.


## Putting LATS pieces together

First, we need to define the search tree nodes. Then write the six operations which are the building blocks of LATS.

| | piece | what it does |
|---|---|---|
| Search Tree Nodes | `Node` | one state in the tree, with `N`, `V`, and `UCT` |
| 1. Selection | `descend` | walk down to the node this attempt works on |
| 2. Expansion | `expand` | sample `n` actions, make all `n` children |
| 3. Evaluation | `compute_V` | `V(s) = lam LM(s) + (1 - lam) SC(s)`, called from inside `expand` |
| 4. Simulation | `simulate` | walk the highest-`V` branch until the trajectory ends |
| 5. Backpropagation | `backprop` | carry the environment's reward up the tree |
| 6. Reflection | `reflect` | write a sentence about a failure, for the next attempt |

You write them in sections 2 through 8, then put the six operations into one
loop in section 9.

First load the hyperparameters that control LATS. The functions between an agent and the environment are already loaded from the greedy warm-up.

In [11]:
# @title Given: The Knobs that turn the LATS loop { display-mode: "form" }

import math
from collections import Counter

import intranet
from intranet import transition

# ==== the knobs the lecture named ==================================================
#
# Imported rather than retyped. agent_lib.py holds exactly one definition of each, and
# the reference implementation this module is checked against reads those same knobs, so
# nothing here can drift away from the numbers in the lecture.

from agent_lib import LAM, N_DEFAULT, SC_VOTES, SUCCESS, W, Memory

#   W = 1.0          the exploration weight in UCT. LATS uses w = 1
#   LAM = 0.5        V(s) = lam LM(s) + (1 - lam) SC(s). The paper's setting on the
#                    question answering task this notebook is modelled on
#   N_DEFAULT = 3    actions sampled per expansion (the paper samples 5)
#   SC_VOTES = 4     independent draws that SC(s) is measured over
#   SUCCESS = 0.9    a reward at or above this counts as solved, and stops the search
#
# LM, the frozen model, was built once in section 1 and is NOT rebuilt here. One object,
# and every operation below calls it. Nothing about it is ever updated.


def at(query_idx, actions):
    """GIVEN. The state you reach by playing `actions` from the portal. Used all through."""
    state = intranet.initial_state(query_idx)
    for action in actions:
        state, _, _, done, info = transition(state, action)
        assert not info["illegal"], "%s: %s" % (action, info["reason"])
    return state


BRIDGE = ["goto_site[people]", "next", "next"]              # people / Team directory
TABLE = ["goto_site[facilities]", "next", "follow[1]"]      # facilities / the lounges

print("knobs loaded.  w = %.1f   lam = %.1f   n = %d   SC votes = %d   success = %.1f"
      % (W, LAM, N_DEFAULT, SC_VOTES, SUCCESS))


knobs loaded.  w = 1.0   lam = 0.5   n = 3   SC votes = 4   success = 0.9


## 2. 🎯 The Node

A `Node` is one state in the search tree, plus what the search remembers about it: the
state itself, its parent, the action that produced it, its children, a visit count `N`,
and a value `V`. `N` starts at 1 so that $\log N(p)$ is defined on the first pass.

The constructor is given. What you implement is **UCT**, the score selection uses when it
picks which child to walk into:

$$\mathrm{UCT}(s) \;=\; V(s) \;+\; w \sqrt{\frac{\log N(p)}{N(s)}}$$

where $V(s)$ is exploitation: how good this node looks so far, $w$ is exploration weight (here w = 1) $N(s)$ and $N(p)$ represent how many attempts have passed through this node and its parent respectively, and $\sqrt{\log N(p)\,/\,N(s)}$ is exploration: large when this child has been tried little relative to its parent. The first term prefers a branch that has paid off. The second prefers a branch that has hardly been tried. `w` is the tradeoff between them.


In [14]:
class Node:
    """One state in the tree, plus what the search keeps about it."""

    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = []
        self.N = 1
        self.V = 0.0
        self.eval_value = 0.0

    def UCT(self, w):
        # 🎯🎯 Define exploit as V(s), and explore as sqrt(log N(p) / N(s)).
        exploit = self.V
        # Assuming parent is not None for UCT calculation for children
        # and N values are always >= 1 as per initialization.
        explore = math.sqrt(math.log(self.parent.N) / self.N)
        # 🎯🎯 Define uct as the sum of exploit and w times explore.
        uct = exploit + w * explore
        return uct

    def compute_V(self, mem=None, lam=LAM):
        """Operation 3, Evaluation. Written in the Evaluation section."""
        raise NotImplementedError(
            "compute_V is the Evaluation section. Fill that gap in and run its "
            "cell: its last line attaches the method to this class.")

In [15]:
# @title Checker for exercise 2 { display-mode: "form" }
from checkers import check_node
print("checker for exercise 2 loaded")
check_node(Node)

checker for exercise 2 loaded
checking your Node class ...

  OK  state, parent, action, children, N = 1, V = 0.0, eval_value = 0.0
  OK  a child remembers its parent and the action that made it
  OK  each Node gets its own children list
  OK  UCT matches the formula, 1.02919
  OK  w = 0 leaves V alone, so the second term really is weighted by w
  OK  before anything is tried UCT is V exactly, because log(1) = 0
  OK  between equal V, the less visited child scores higher

PASSED  Exercise, the Node class


True

## 3. 🎯 Selection: Descending the Tree

To **descend** down the tree, you will use the provided `best_child_by_uct` function to pick the most promising path at every level.

Here is the step-by-step logic:

* **Keep Going:** If the current node has children, pick the best one and call `descend` on it recursively to keep moving down.
* **Stop at the Edge:** As soon as you hit a node with *no* children, stop! This is your frontier—the exact spot where this current attempt will get to work. Simply return this node.


In [ ]:
# @title Given: best_child_by_uct

def best_child_by_uct(s, w):
    """Given. The child of `s` with the highest UCT score."""
    best_child = s.children[0]
    best_uct = best_child.UCT(w)
    for c in s.children:
        if c.UCT(w) > best_uct:
            best_uct = c.UCT(w)
            best_child = c
    return best_child

In [ ]:
def descend(s, w):
    """Operation 1 of 6, Selection. Walk down to the node this attempt will work on."""
    if len(s.children) > 0:
        # 🎯🎯 Select the best child
        best = ???
        # 🎯🎯 Descend into the best child by calling descend recursively
        return ???
    else:
        return s

In [ ]:
# @title Checker for exercises 3 { display-mode: "form" }
from checkers import check_descend
print("checker for exercise 3 loaded")
check_descend(descend)

## 4. 🎯 Expansion

This is the only place where our search tree grows, and it always sprouts `n_actions` new child nodes all at the exact same time.

**Why grow everything all at once?**
It comes down to how quickly we can judge each new option. The moment a new child node is created, a function called `compute_V` instantly assigns it a score. Because all `n` actions can be scored and compared immediately, there is no benefit to holding any of them back or creating them one by one. We just generate the whole batch!

**How it looks in the code:**
Because of this instant scoring, **Evaluation** (Operation 3) actually lives *inside* **Expansion** (Operation 2).
In the code loop below, a single line—`child.compute_V(mem)`—handles the entire Evaluation step. (You'll write the actual method for this in the next section.)

The policy `propose_actions` is already provided for you. Loop through all the actions produced by the policy and expand the tree to its child nodes.

In [ ]:
# @title Given: the policy
def propose_actions(state, n_actions, mem):
    """GIVEN. The policy. One model call, n_actions candidate actions, SAMPLED."""
    if MAX_STEPS - state["steps"] <= 1:
        answers = LM.propose(state, n=n_actions, memory=mem, answers_only=True)
        if answers:
            return answers
    return LM.propose(state, n=n_actions, memory=mem)


print("the policy is loaded. Expansion is the only operation that calls it")


In [ ]:
def expand(s, n_actions, mem):
    """Operation 2 of 6, Expansion, with 3, Evaluation, inside it."""
    if is_terminal(s.state) or len(s.children) > 0:
        return                             # given: answered, or already expanded

    state = s.state
    # 🎯🎯 Get the actions from the policy.
    actions = ???

    for action in actions:
        # 🎯🎯 Play the action using the step function like in the previous exercise 1.
        ???, ???, ???, ??? = step(???, ???)
        if info["illegal"]:
            continue
        # 🎯🎯 Create a new node with the new state, the current node as parent, and the action that got us here.
        child = ???
        # 🎯🎯 Score the child using the compute_V method.
        ???
        # 🎯🎯 Add the child to the current node's children list.
        ???

In [ ]:
# @title Checker for exercise 4 { display-mode: "form" }
from checkers import check_expand
print("checker for exercise 4 loaded")
check_expand(expand)

## 5. 🎯 Evaluation

Expansion just called `child.compute_V(mem)` on every new child. That call is the whole of
operation 3. This section writes the method.

A game search would find out what a child is worth by playing the game out at random.
There is no cheap random way to finish "find the right intranet page", so LATS asks the
model instead:

$$V(s) \;=\; \lambda \cdot \mathrm{LM}(s) \;+\; (1 - \lambda)\cdot \mathrm{SC}(s)$$

Two terms, because one would not be enough. `LM(s)` is the model shown the trajectory so
far and asked how promising it looks, which is the model marking its own homework.
`SC(s)` is a second opinion of a different kind: sample the next action several times over
and count how often the model says the same thing. That catches a state the model is
confused about even when it is happy to call it good.

`SC(s)` is given, and the cell below prints both halves on the two pages question 1 turns
on: the team directory, which is the bridge, and the lounges table, which looks perfect
and is one fact short.


In [ ]:
# @title Given: SC(s), and the two halves on two real pages
def self_consistency(state, mem=None):
    """GIVEN. SC(s): the share of independent draws that name the same next action."""
    votes = []
    for i in range(SC_VOTES):
        picked = LM.propose(state, n=1, draw=i + 1, memory=mem)
        if picked:
            votes.append(picked[0])
    if not votes:
        return 0.0
    return Counter(votes).most_common(1)[0][1] / len(votes)

In [ ]:
def compute_V(self, mem=None, lam=LAM):
    """Operation 3 of 6, Evaluation. V(s) = lam LM(s) + (1 - lam) SC(s)."""
    if is_terminal(self.state):
        self.V = reward_of(self.state)         # given: the truth, not a guess
    else:
        # 🎯🎯 Get the LM(s) score using the LM.value method
        lm_score = ???
        # 🎯🎯 Get the SC(s) score using the self_consistency method
        sc_score = ???
        # 🎯🎯 Compute the value as a weighted sum of the LM(s) and SC(s) scores using the given lambda (lam)
        self.V = ???

    self.eval_value = self.V     # given: a frozen copy of the guess, for the tables later
    return self.V


Node.compute_V = compute_V       # given: attach it to the class from exercise 2

In [ ]:
# @title Checker for exercise 5 { display-mode: "form" }
from checkers import check_compute_V
print("checker for exercise 5 loaded")
check_compute_V(compute_V)

In [ ]:
# @title Demonstration of V(s) on Question1 { display-mode: "form" }

print(intranet.QUERIES[0].question)
print()
for label, actions in (("the team directory, in people", BRIDGE),
                       ("the coffee lounge table, in facilities", TABLE)):
    state = at(0, actions)
    print("  %-38s  LM(s) = %.3f   SC(s) = %.3f"
          % (label, LM.value(state), self_consistency(state)))

table = at(0, TABLE)
print()
print("  every row of that second page:")
for line in intranet.page_at(table["site"], table["page"]).body:
    if ": " in line:
        print("      " + line)
print()
print("  the answers the policy likes best there, in its own order:")
for action in LM.propose(table, n=3, greedy=True, answers_only=True):
    print("      " + action)

V(s) is the model’s own guess of “how good is this state?”, before any real reward. On the lounge table that guess is high, because:

LM(s): the page looks like the answer page (rooms, lounges, table of codes).
SC(s): when you resample the next action, the model keeps proposing the same kind of move — answer with a row from this table.
So the search thinks this node is promising. It is still not enough to be right: four plausible answers, and the model has no way to pick the correct answer unless it already visited the people page. Guessing a row from here is 0.000 three out of four times, only N-310 pays 1.000.

## 6. 🎯 Simulation

Expansion scored every new child. Simulation keeps going from there: expand and evaluate
again at the next depth, always stepping into the child with the largest `V`, until the
trajectory ends.

It follows `V` alone, with no exploration term using the function `best_child_by_value` given below. The walk is trying to finish this attempt well, not to learn about the rest of the tree. A random walk over the intranet
would answer nonsense, and nonsense is worth 0.000 whatever page it stopped on.


In [ ]:
# @title Given: best_child_by_value

def best_child_by_value(s):
    """GIVEN. The child with the largest V, with NO exploration term."""
    best_child = s.children[0]
    for c in s.children:
        if c.V > best_child.V:
            best_child = c
    return best_child

In [ ]:
def simulate(s, n_actions, mem):
    """Operation 4 of 6, Simulation. Walk on, expanding and evaluating at every depth."""

    if is_terminal(s.state):
        return s
    expand(s, n_actions, mem) #given: expands the current node
    if len(s.children) == 0:
        return s
    # 🎯🎯 Get the best child using the best_child_by_value function
    best = ???
    # 🎯🎯 Recursively simulate the best child
    return ???

In [ ]:
# @title Checker for exercise 6 { display-mode: "form" }
from checkers import check_simulate
print("checker for exercise 6 loaded")
check_simulate(simulate)

## 7. 🎯 Backpropagation

When the simulation finishes a path, it hands back the final node. Up until this point, every score (`V`) in the tree has just been the model's *guess*.

**Backpropagation** is where reality kicks in. The environment grades the final answer and hands out an actual reward. We then take that real reward and pass it all the way back up the chain to the starting point (the root), updating the average score for every single step we took to get there.

Earlier, `compute_V` gave each node an initial guess, which counts as your first data point ($N = 1$). When the real reward ($r$) arrives, we increase the visit count ($N$) by 1, and update the node's value ($V$) by averaging the new reward into the old score using this formula:

$$V = \frac{V \times (N - 1) + r}{N}$$

As this is a single-player game, the same reward (`r`) is used at every level as it travels back up the tree. In a competing game, the reward (`r`) would get flipped.

In [ ]:
def backprop(node, r):
    """Operation 5 of 6, Backpropagation. The same r to every node up the chain."""

    node.N = node.N + 1
    # 🎯🎯 Update the node's value by averaging the new reward into the old score
    node.V = ???
    if node.parent is not None:
        # 🎯🎯 Recursively backpropagate the reward to the parent node
        ???

In [ ]:
# @title Checker for exercises 7 { display-mode: "form" }
from checkers import check_backprop
print("checker for exercise 7 loaded")
check_backprop(backprop)

## 8. 🎯 Reflection

The reward is officially logged! But if the agent didn't successfully solve the mystery, one final thing happens before it gets to try again.

**Leaving a Note to Self:**
The model is shown exactly what it just did and asked, *"What should we do differently next time?"* It then writes a quick sentence of advice. This little note gets stashed in a memory buffer, and the failed answer gets penalized so the agent is less likely to repeat the exact same mistake.

**No Weight Training Required!**
This is a huge point: **no model weights are moved or updated.** The *only* thing that carries over from a failed attempt to the next try is plain old language. It learns just by reading its own notes!

**Seeing the Big Picture:**
To write a helpful reflection, the model needs to see the *entire* journey it just took, not just the final dead end. This operation is the one and only place in the entire notebook that needs the full trajectory, which is why the `path_to_root` function is already given to you.

In [ ]:
# @title Given: path_to_root
def path_to_root(node):
    """GIVEN. The actions from the root down to `node`, root first."""
    if node.parent is None:
        return []
    return path_to_root(node.parent) + [node.action]

In [ ]:
def reflect(leaf, r, mem):
    """Operation 6 of 6, Reflection. The only operation that needs the whole path."""

    if mem is None or answer_of(leaf.state) is None:
        return
    # 🎯🎯 Reflect the leaf node using the LM.reflect method, passing the leaf state, the reward, and the path to the root
    note = ???
    mem.add(note, leaf.state, r)    #given: add the note to the memory buffer
    return note

In [ ]:
# @title Checker for exercise 8 { display-mode: "form" }
from checkers import check_reflect
print("checker for exercise 8 loaded")
check_reflect(reflect)

## 9. 🎯 Put the Operations Together

Every piece is written and checked. Wire them into the LATS loop.

| | piece | in a few words | yours since |
|---|---|---|---|
| Node | `class Node` | one state, plus visits `N` and value `V` | exercise 2 |
| 1. Selection | `descend(root, exploration_weight)` | walk to a leaf by UCT | exercise 3 |
| 2. Expansion | `expand(leaf, n_actions, memory)` | grow children from the model's next actions | exercise 4 |
| 3. Evaluation | `compute_V` inside `expand` | score each new child | exercise 5 |
| 4. Simulation | `end = simulate(leaf, n_actions, memory)` | follow highest `V` until an answer | exercise 6 |
| 5. Backpropagation | `backprop(end, reward)` | average the real reward up the path | exercise 7 |
| 6. Reflection | `reflect(end, reward, memory)` | note what went wrong for later attempts | exercise 8 |

One attempt is those six operations in order, written onto Nodes. Selection picks a leaf in the tree you already have. Expansion and Evaluation grow it: the model proposes children and guesses how good they look. Simulation walks from there to a finish. Then the environment pays a `reward`. If that is not a success, Reflection stores a note and Backpropagation writes the reward into every `V` on the path, so the next Selection sees a better ranking. Repeat until a success or the attempt budget runs out.

The language model is frozen. Only `reward` is evidence from outside it.


In [ ]:
# @title Given: the bookkeeping around your search { display-mode: "form" }
import textwrap
import time

import matplotlib.pyplot as plt

from agent_lib import K_DEFAULT
from checkers import _restore, _watching

#   K_DEFAULT = 8     attempts per question (the paper runs 50)

def most_visited_child(s):
    """GIVEN. The child the search spent the most attempts on.

    What `lats` commits to when the budget runs out without a success. Most visited, not
    highest V: a high mean off one lucky attempt is noise, while a high visit count
    means the search kept coming back.
    """
    best_child = s.children[0]
    for c in s.children:
        if c.N > best_child.N:
            best_child = c
    return best_child


def most_visited_leaf(root):
    """GIVEN. Follow the most visited child down until there is nothing below it.

    The pseudocode's last line reads `answer_of(most_visited_child(root).state)`, and
    that is a slip in the file: the most visited CHILD of the root is one action in, so
    nothing has been answered on it. What the search commits to when its budget runs out
    is the whole trajectory it kept coming back to, which means following the visit
    counts all the way down.
    """
    if len(root.children) == 0:
        return root
    return most_visited_leaf(most_visited_child(root))


def replay(query_idx, actions):
    """GIVEN. Play a list of actions in a fresh episode and hand back where it ended.

    Used to make the search's own claims checkable: whatever it says it scored, this is
    the environment paying for that trajectory again from the portal.
    """
    state, reward = initial_state(query_idx), 0.0
    for action in actions:
        state, _, reward, _, _ = transition(state, action)
    return state, float(reward)


def count_nodes(root):
    """GIVEN. How big the tree got."""
    total, stack = 0, [root]
    while stack:
        node = stack.pop()
        total += 1
        stack.extend(node.children)
    return total


def run(query_idx, n=N_DEFAULT, k=K_DEFAULT, w=W, use_reflection=True):
    """GIVEN. YOUR `lats`, run on one question, with an observer around it.

    Nothing in here is search. The eleven lines that matter are the ones you wrote; this
    watches five of them being called, and hands back what the tables below need: the
    tree, one row per attempt, the answer the search committed to, and what the
    environment pays for that answer when it is replayed from the portal.

    `use_reflection=False` passes no buffer at all, which switches operation 6 off.

    Returns {"answer", "reward", "trajectory", "attempts", "calls", "passes", "nodes",
             "root", "mem", "seconds"}.
    """
    seen = {"root": None, "rows": []}

    def record(name, args, result):
        if name == "descend":
            if seen["root"] is None:
                seen["root"] = args[0]
            seen["rows"].append({"leaf": None})
        elif name == "simulate" and seen["rows"]:
            seen["rows"][-1]["leaf"] = result

    mem = Memory() if use_reflection else None
    LM.calls = 0
    passes_before = getattr(LM, "forward_passes", 0)
    started = time.time()

    real = _watching(("descend", "simulate"), record)
    try:
        answer = lats(initial_state(query_idx), k, n, w, mem)
    finally:
        _restore(real)

    attempts = []
    for i, row in enumerate(seen["rows"], start=1):
        leaf = row["leaf"]
        if leaf is None:                      # the gap is unfilled, or it never simulated
            continue
        attempts.append({"attempt": i, "reward": reward_of(leaf.state),
                         "answer": answer_of(leaf.state),
                         "trajectory": path_to_root(leaf), "leaf": leaf})

    if attempts and attempts[-1]["reward"] >= SUCCESS:
        committed = attempts[-1]["leaf"]
    else:
        committed = most_visited_leaf(seen["root"])
    trajectory = path_to_root(committed)
    _, reward = replay(query_idx, trajectory)

    if answer_of(committed.state) != answer:
        print("  note: lats returned %r and the trajectory it committed to answered %r"
              % (answer, answer_of(committed.state)))

    return {"answer": answer, "reward": reward, "trajectory": trajectory,
            "attempts": attempts, "calls": LM.calls, "nodes": count_nodes(seen["root"]),
            "passes": getattr(LM, "forward_passes", 0) - passes_before,
            "root": seen["root"], "mem": mem, "seconds": time.time() - started}


SCOPE = list(range(len(QUERIES)))       # all ten questions

print("bookkeeping loaded. %d questions in scope, on %s"
      % (len(SCOPE), type(LM).__name__))

In [ ]:
def lats(root_state, n_attempts, n_actions, exploration_weight, memory):
    """The whole search: the six operations, n_attempts times over."""
    root = Node(root_state, None) # given: create a root node

    for _ in range(n_attempts):
        # 🎯🎯 Select the best child using the descend function
        leaf = ???
        # 🎯🎯 Expand the selected child using the expand function
        ???
        # Evaluation is done inside the expand function
        # 🎯🎯 Simulate the expanded child using the simulate function
        end = ???

        reward = reward_of(end.state)                # given: the number from the environment
        if reward >= SUCCESS:
            return answer_of(end.state)              # success stops the search

        # 🎯🎯 Reflect the simulated child using the reflect function
        ???
        # 🎯🎯 Backpropagate the reward to the parent node using the backprop function
        ???

    # Given: returns the answer of the most visited leaf node
    return answer_of(most_visited_leaf(root).state)

In [ ]:
# @title Checker for exercise 9 { display-mode: "form" }
from checkers import check_lats
print("checker for exercise 9 loaded")
check_lats(lats)

## 10. Greedy against LATS

One question proves nothing. The same frozen model now answers all questions both ways.

In [ ]:
# @title Load functions to run greedy and LATS comparison { display-mode: "form" }

SUITES = {}              # (n, k, scope) -> the rows, so a setting is never run twice


def measure(n=N_DEFAULT, k=K_DEFAULT, scope=None):
    """Run one setting of your search over `scope`, or over every question in SCOPE."""
    scope = tuple(SCOPE if scope is None else scope)
    if (n, k, scope) not in SUITES:
        SUITES[(n, k, scope)] = [run(i, n=n, k=k) for i in scope]
    rows = SUITES[(n, k, scope)]
    m = len(rows)
    return {"rows": rows,
            "reward": sum(r["reward"] for r in rows) / m,
            "success": sum(r["reward"] >= SUCCESS for r in rows) / m,
            "calls": sum(r["calls"] for r in rows) / m}


def blind_line(scope):
    """The line to beat, and it is not zero.

    Every hop 2 page is a table keyed by the thing hop 1 reveals, so an agent that
    reaches the right page and guesses a row scores 1 / (number of rows) in
    expectation. This reads the query set's own bookkeeping, which is the grader's
    book and not anything the agent can see.
    """
    return sum(1.0 / (1 + len(QUERIES[i].decoy_answers)) for i in scope) / len(scope)


BLIND = blind_line(SCOPE)


def suite_table(scope, n=N_DEFAULT, k=K_DEFAULT):
    """Greedy against your search on `scope`: one row as each question finishes."""
    scope = list(scope)
    started = time.time()
    baseline, rows = [], []
    total = len(scope)
    key = (n, k, tuple(scope))
    cached = SUITES.get(key)

    try:
        from tqdm.auto import tqdm
        questions = tqdm(scope, desc="questions", unit="q", dynamic_ncols=True)
        log = questions.write
    except Exception:
        questions = scope
        log = lambda msg: print(msg, flush=True)

    print("      %-24s %6s    %-24s %6s"
          % ("greedy answer", "F1", "LATS answer", "F1"), flush=True)

    for step, i in enumerate(questions, start=1):
        if not hasattr(questions, "write"):
            filled = int(20 * (step - 1) / total) if total else 0
            bar = "[" + "#" * filled + ">" + "-" * (20 - filled - 1) + "]"
            elapsed = time.time() - started
            if step == 1:
                eta = "timing after Q1"
            else:
                left = (elapsed / (step - 1)) * (total - step + 1)
                eta = "~%.0fs left" % left
            print("  %s  %d/%d  Q%d running ...  %s"
                  % (bar, step, total, i + 1, eta), flush=True)

        q0 = time.time()
        g = greedy(i)                          # YOUR greedy, from the warm up
        s = cached[step - 1] if cached is not None else run(i, n=n, k=k)
        took = time.time() - q0
        baseline.append(g)
        rows.append(s)
        log("  Q%-3d  %-24s %6.3f    %-24s %6.3f   (%.0fs)"
            % (i + 1, str(g["answer"])[:24], g["reward"],
               str(s["answer"])[:24], s["reward"], took))

    SUITES[key] = rows
    searched = {"rows": rows,
                "reward": sum(r["reward"] for r in rows) / total,
                "success": sum(r["reward"] >= SUCCESS for r in rows) / total,
                "calls": sum(r["calls"] for r in rows) / total}

    print("\n  %-28s %8s %9s %8s" % ("", "F1", "solved", "calls"))
    for label, block in (("greedy, one trajectory", baseline),
                         ("LATS", searched["rows"])):
        m = len(block)
        print("  %-28s %8.3f %6d/%-2d %8.1f"
              % (label, sum(r["reward"] for r in block) / m,
                 sum(r["reward"] >= SUCCESS for r in block), m,
                 sum(r["calls"] for r in block) / m))
    print("  %-28s %8.3f" % ("blind, right page wrong row", blind_line(scope)))
    print("\n  %.0f seconds for %d question%s"
          % (time.time() - started, total, "" if total == 1 else "s"))
    return baseline, searched


print("suite_table(scope) is loaded. It runs greedy and your search side by side")


Do a quick run and check if the loaded functions work on the first question. The summary at the bottom of the next run also prints `blind, right page wrong row`: the expected F1 if you reach the right table and guess a row. That is the floor to beat, and it is not zero.

In [ ]:
QUICK = SCOPE[1:2]                 # Run on a random question

started = time.time()
greedy_quick, searched_quick = suite_table(QUICK)

per_question = (time.time() - started) / len(QUICK)
print("  about %.0f seconds per question on this machine, so the whole suite below"
      % per_question)
print("  will take roughly %.0f minutes" % (per_question * len(SCOPE) / 60.0))


The next cell opens one of the trees from the quick run above, before the full suite.

Each first action from the root has two scores:

- **`guessed`** — what `compute_V` said when that child was created. That is the frozen model's opinion of how promising the page looks. It is never updated.
- **`V(s) now`** — the running average of the *real* rewards the environment later paid on attempts that went down that branch. Backpropagation writes this number.

If the two columns match, search has not yet learned anything the model did not already believe. If they differ, the reward has rewritten the ranking. The language model did not change its mind — its weights are frozen. Only the tree's `V` was overwritten.


In [ ]:
# @title Effect of backpropagation on the tree { display-mode: "form" }

picked = None
for row in searched_quick["rows"]:
    if len(row["attempts"]) > 1:
        picked = row
        break
if picked is None:
    picked = searched_quick["rows"][0]
    print("Every question above was solved on the first attempt, so nothing was ever")
    print("backpropagated: `lats` returns on success, before backprop. The two value")
    print("columns will agree, and that is the early stop and not a bug.\n")

root = picked["root"]
print("the root's options after %d attempt%s\n"
      % (len(picked["attempts"]), "" if len(picked["attempts"]) == 1 else "s"))
print("  %-24s %5s %9s %9s %7s"
      % ("first action", "N(s)", "guessed", "V(s) now", "UCT"))
for child in sorted(root.children, key=lambda c: -c.N):
    print("  %-24s %5d %9.3f %9.3f %7.3f"
          % (child.action, child.N, child.eval_value, child.V, child.UCT(W)))
print("\n  guessed  is what compute_V said, before anything had been played")
print("  V(s) now is the running mean of what the environment actually paid")

if picked["mem"] is not None and picked["mem"].notes:
    print("\nand what the critic wrote to itself between those attempts\n")
    for note in picked["mem"].notes:
        print(textwrap.fill(note, 74, initial_indent="  ", subsequent_indent="  "))


If those three look right, run the whole set.


In [ ]:
greedy_rows, searched = suite_table(SCOPE)

That is the exercise. Greedy has one trajectory and no way back, so it commits to the
first plausible page and answers off it; the search spends more model calls to try an
answer, read what the environment paid, and go somewhere else. The gap between those two
rows is what the tree bought, and the *blind* line underneath is the reminder that
reaching the right page was never the hard part.

Nothing in it was trained. One frozen model proposed every action, scored every state and
wrote every reflection, and the only number that did not come out of that model is the
reward the environment paid at the end of each attempt.